# Cinemática Inversa de un Brazo Robótico de 3 Eslabones

### Caso 2: Cinemática Inversa en Tiempo Real (3 Articulaciones)
* **Objetivo:** Calcular los ángulos de los servomotores $(\theta_1, \theta_2, \theta_3)$ para posicionar el actuador final en $(x_{\text{target}}, y_{\text{target}})$.
* **Problema computacional:** Las transformaciones trigonométricas acopladas vuelven prohibitivo el cálculo simbólico en bucles de control a 60 Hz.
* **Mecanismo aplicado:** Algoritmo cuasi-Newton con cota máxima de paso (`max_step = 0.5 rad`) para prevenir singularidades y latigazos mecánicos.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ==========================================================
# 1. Parámetros del Robot (3 eslabones, alcance total: 3.5 m)
# ==========================================================
L1, L2, L3 = 1.5, 1.2, 0.8

# Meta lejana en el cuadrante superior izquierdo
x_target = 0.8
y_target = 2.4
phi_target = np.radians(60.0)  # Pinza apuntando a 60°
target_vector = np.array([x_target, y_target, phi_target])

def forward_kinematics(theta):
    t1, t2, t3 = theta[0], theta[1], theta[2]

    x1 = L1 * np.cos(t1)
    y1 = L1 * np.sin(t1)

    x2 = x1 + L2 * np.cos(t1 + t2)
    y2 = y1 + L2 * np.sin(t1 + t2)

    x3 = x2 + L3 * np.cos(t1 + t2 + t3)
    y3 = y2 + L3 * np.sin(t1 + t2 + t3)

    phi = t1 + t2 + t3
    return np.array([x3, y3, phi]), [(0, 0), (x1, y1), (x2, y2), (x3, y3)]

def F_robot(theta):
    actual, _ = forward_kinematics(theta)
    return actual - target_vector

def jacobiano_inicial(F, x0, h=1e-5):
    n = len(x0)
    J = np.zeros((n, n))
    Fx0 = F(x0)
    for j in range(n):
        x_step = np.copy(x0)
        x_step[j] += h
        J[:, j] = (F(x_step) - Fx0) / h
    return J

# ==========================================================
# 2. Broyden con Control de Paso (Movimiento Suave)
# ==========================================================
def resolver_broyden_suave(theta_init, tol=1e-4, max_iter=15, max_step=0.5):
    theta = np.array(theta_init, dtype=float)
    historia = [theta.copy()]

    B = jacobiano_inicial(F_robot, theta)

    for _ in range(max_iter):
        Fx = F_robot(theta)
        error_dist = np.linalg.norm(Fx[:2])

        if error_dist < tol:
            break

        # Paso propuesto por Broyden
        s = np.linalg.solve(B, -Fx)

        norma_s = np.linalg.norm(s)
        if norma_s > max_step:
            s = s * (max_step / norma_s)

        theta_new = theta + s

        # Actualización de Broyden sobre el paso realmente ejecutado
        y_diff = F_robot(theta_new) - Fx
        B = B + np.outer(y_diff - B @ s, s) / np.dot(s, s)

        theta = theta_new
        historia.append(theta.copy())

    return historia

# Posición inicial: extendido abajo a la derecha (10°, 15°, 10°)
theta_inicial = np.radians([10.0, 15.0, 10.0])
historia_angulos = resolver_broyden_suave(theta_inicial)

# ==========================================================
# 3. Animación en Jupyter
# ==========================================================
fig, ax = plt.subplots(figsize=(7, 7))

# Objetivo y flecha de orientación
ax.scatter([x_target], [y_target], color='#dc2626', s=180, zorder=6, label='Objetivo (Target)')
dx_flecha = 0.3 * np.cos(phi_target)
dy_flecha = 0.3 * np.sin(phi_target)
ax.arrow(x_target, y_target, dx_flecha, dy_flecha, color='#dc2626', width=0.03, head_width=0.09, zorder=6)

# Elementos móviles
linea_brazo, = ax.plot([], [], '-o', color='#2563eb', lw=4, markersize=8, mfc='white', mew=2, label='Brazo (3 eslabones)')
trazo_punta, = ax.plot([], [], ':', color='#0284c7', lw=2, alpha=0.7, label='Trayectoria de la pinza')
texto_iter = ax.text(0.04, 0.88, '', transform=ax.transAxes, fontsize=10,
                     bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#cbd5e1'))

ax.set_xlim(-0.8, 3.6)
ax.set_ylim(-0.3, 3.2)
ax.set_aspect('equal')
ax.grid(True, linestyle=':', alpha=0.6)
ax.set_title('Convergencia Cinemática con Broyden Regulado', fontsize=12, fontweight='bold')
ax.set_xlabel('Coordenada X (m)')
ax.set_ylabel('Coordenada Y (m)')
ax.legend(loc='lower left', framealpha=0.9)

# Precalcular las posiciones de la pinza para un trazo limpio
todas_puntas = [forward_kinematics(th)[1][-1] for th in historia_angulos]

def update_brazo(frame):
    th = historia_angulos[frame]
    _, articulaciones = forward_kinematics(th)

    xs = [pt[0] for pt in articulaciones]
    ys = [pt[1] for pt in articulaciones]
    linea_brazo.set_data(xs, ys)

    pts_recorridos = todas_puntas[:frame+1]
    trazo_punta.set_data([p[0] for p in pts_recorridos], [p[1] for p in pts_recorridos])

    dist_error = np.linalg.norm(np.array([xs[-1], ys[-1]]) - np.array([x_target, y_target]))
    texto_iter.set_text(
        f"Iteración: {frame} / {len(historia_angulos)-1}\n"
        f"Distancia a meta: {dist_error:.4f} m\n"
        f"Estado: {'¡OBJETIVO ALCANZADO!' if dist_error < 0.01 else 'Ajustando articulaciones...'}"
    )
    return linea_brazo, trazo_punta, texto_iter

anim_robot = FuncAnimation(fig, update_brazo, frames=len(historia_angulos), interval=900, blit=True)
plt.close(fig)

HTML(anim_robot.to_jshtml())